In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATA_DIR="/content/drive/MyDrive/drug_repurposing/data"

In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.1 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GATConv, HeteroConv
from torch_geometric.transforms import RandomLinkSplit

from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
DATA_DIR = "/content/drive/MyDrive/drug_repurposing/data"

checkpoint = torch.load(f"{DATA_DIR}/processed/filtered_graph.pt")

data = checkpoint["data"]
node_maps = checkpoint["node_maps"]
reverse_node_maps = checkpoint["reverse_node_maps"]

print(data)

In [ ]:
hidden_dim = 128

for node_type in data.node_types:
    if 'x' not in data[node_type]:
        data[node_type].x = torch.randn(data[node_type].num_nodes, hidden_dim)

In [ ]:
torch.manual_seed(42)

split = RandomLinkSplit(
    num_val=0.15,
    num_test=0.15,
    is_undirected=False,
    add_negative_train_samples=True,
    neg_sampling_ratio=1.0,
    edge_types=('Compound', 'CtD', 'Disease'),
)

train_data, val_data, test_data = split(data)

In [ ]:
class HeteroRGAT(nn.Module):
    def __init__(self, hidden_dim, heads=4):
        super().__init__()

        # Relation-specific GAT layers
        self.convs = nn.ModuleDict({
            str(edge_type): GATConv((-1, -1), hidden_dim, heads=heads, concat=False)
            for edge_type in data.edge_types
        })

        # Learnable relation weights ⭐
        self.rel_weights = nn.ParameterDict({
            str(edge_type): nn.Parameter(torch.tensor(1.0))
            for edge_type in data.edge_types
        })

    def forward(self, x_dict, edge_index_dict):

        out_dict = {key: torch.zeros_like(x_dict[key]) for key in x_dict}

        for edge_type in edge_index_dict:
            src, rel, dst = edge_type

            conv = self.convs[str(edge_type)]
            weight = self.rel_weights[str(edge_type)]

            out = conv(x_dict, edge_index_dict[edge_type])
            out = F.relu(out)

            out_dict[dst] += weight * out

        return out_dict

In [ ]:
def decode(z_src, z_dst):
    return torch.sum(z_src * z_dst, dim=-1)

In [ ]:
device = torch.device("cpu")

model = HeteroRGAT(hidden_dim).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.005,
    weight_decay=5e-4
)

criterion = nn.BCEWithLogitsLoss()

In [ ]:
def train():

    model.train()
    optimizer.zero_grad()

    z_dict = model(train_data.x_dict, train_data.edge_index_dict)

    edge_store = train_data['Compound', 'CtD', 'Disease']

    src = edge_store.edge_label_index[0]
    dst = edge_store.edge_label_index[1]
    labels = edge_store.edge_label.float()

    z_src = z_dict["Compound"][src]
    z_dst = z_dict["Disease"][dst]

    preds = decode(z_src, z_dst)

    loss = criterion(preds, labels)

    loss.backward()
    optimizer.step()

    return loss.item()

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

@torch.no_grad()
def evaluate(data_split):

    model.eval()

    z_dict = model(data_split.x_dict, data_split.edge_index_dict)

    edge_store = data_split['Compound', 'CtD', 'Disease']

    src = edge_store.edge_label_index[0]
    dst = edge_store.edge_label_index[1]
    labels = edge_store.edge_label.cpu().numpy()

    z_src = z_dict["Compound"][src]
    z_dst = z_dict["Disease"][dst]

    preds = decode(z_src, z_dst).cpu().numpy()

    auc = roc_auc_score(labels, preds)
    ap = average_precision_score(labels, preds)

    return auc, ap

In [ ]:
num_epochs = 50

train_losses = []
val_aucs = []
val_aps = []

for epoch in range(1, num_epochs + 1):

    loss = train()
    val_auc, val_ap = evaluate(val_data)

    train_losses.append(loss)
    val_aucs.append(val_auc)
    val_aps.append(val_ap)

    print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Val AUC: {val_auc:.4f} | Val AP: {val_ap:.4f}")

In [ ]:
test_auc, test_ap = evaluate(test_data)

print("Test AUC:", test_auc)
print("Test AP:", test_ap)

In [ ]:
plt.figure()

plt.plot(train_losses, label="Train Loss")
plt.plot(val_aucs, label="Val AUC")
plt.plot(val_aps, label="Val AP")

plt.xlabel("Epoch")
plt.title("Training Curves (HeteroRGAT)")
plt.legend()

plt.show()

In [ ]:
weights = {k: v.item() for k, v in model.rel_weights.items()}

plt.figure()
plt.bar(range(len(weights)), list(weights.values()))
plt.xticks(range(len(weights)), list(weights.keys()), rotation=90)
plt.title("Learned Relation Importance")
plt.show()

In [ ]:
artifact = {
    "model_name": "hetero_rgat",
    "embedding_dim": hidden_dim,
    "train_losses": train_losses,
    "val_aucs": val_aucs,
    "val_aps": val_aps,
    "test_auc": test_auc,
    "test_ap": test_ap,
    "embeddings": model(data.x_dict, data.edge_index_dict),
    "relation_weights": weights
}

torch.save(artifact, "/content/drive/MyDrive/drug_repurposing/models/hetero_rgat_artifact.pt")

print("Saved RGAT model")